In [48]:
import pandas as pd
from collections import Counter
import math
import ast
import datasets
import os

In [106]:
# Path to the .json generated by the script extract.py
json_file = "/projects/iris/rferreir/GeoBenchmark/data/GeoQuestions1089/dataset.json"

# Path to the output files
output_file = "/projects/iris/rferreir/GeoBenchmark/data/GeoQuestions1089/dataset2.json"

# Path to the directory where to save the HF datasets
output_path = "/projects/iris/rferreir/hub_datasets"

In [114]:
data = pd.read_json(json_file, orient='records')
print(len(data))
data["question_id"] = range(1, len(data) + 1)
data = data[data["cleaned_answer"].apply(len) > 0]
print(len(data))
data.head()

1017
954


,question,cleaned_answer,category,original_answer,answer_type,question_id
0,What is the population of Central Greece?,[570922.0],A,population\r\n570922\r\n,[numeric],1
1,Where is the Dorset county located?,"[[50.8054069086, -2.3225184771]]",A,None,[coord],2
2,Where is Lough Ramor located?,"[[53.8184256149, -7.0822994976]]",A,None,[coord],3
3,What is the population of the Municipality of ...,[38116.0],A,population\r\n38116\r\n,[numeric],4
4,Where is Oxfordshire located?,"[[51.7657655773, -1.3223245468]]",A,None,[coord],5


## Check for coordinates

In [115]:
def check_for_coordinates(row):
    a = row['cleaned_answer']
    if 'coordinates' in row['question'] or 'latitude' in row['question'] or 'longitude' in row['question'] or 'coordinaties' in row['question']:
        print(row['question'])
        print(row['cleaned_answer'])
        print()
    return row

data.apply(check_for_coordinates, axis=1)
print()

What are the geographic coordinates of the USA?
[39.76, -77.0166666667, 39.76, -98.5, 38.8833333333, -77.0166666667, 38.8833333333, -98.5]

What are the coordinaties of the John F. Kennedy International Airport?
[40.6397222222, 73.7788888889]

Give me the latitude and longitude of NYC.
[40.71427, -74.00597]




In [116]:
def change_to_coordinates(row):
    questions = {
        'What are the geographic coordinates of the USA?',
        'What are the coordinaties of the John F. Kennedy International Airport?',
        'Give me the latitude and longitude of NYC.'
    }
    if row['question'] in questions:
        a = row['cleaned_answer'][:2]
        row['cleaned_answer'] = [a]
        row['answer_type'] = ['coord']
    return row

data = data.apply(change_to_coordinates, axis=1)

data.apply(check_for_coordinates, axis=1)
print()

What are the geographic coordinates of the USA?
[[39.76, -77.0166666667]]

What are the coordinaties of the John F. Kennedy International Airport?
[[40.6397222222, 73.7788888889]]

Give me the latitude and longitude of NYC.
[[40.71427, -74.00597]]




## Convert square degrees to square km 

In [117]:
mots = ['area', 'size']

def check_for_degrees(row):
    a = row['cleaned_answer']
    if 'area' in row['question'] or 'size' in row['question'] or 'How large' in row['question']:
        try:
            v = float(a[0])
            if v % 1 != 0:
                row['answer_type'] = ['sq degrees']*len(row['answer_type'])
                print(row['question'], v)
                
        except:
            #print(row['question'])
            pass
    return row

data = data.apply(check_for_degrees, axis=1)

What is the total area of County Galway? 0.0068252496000000005
What is the total area of Glengarra Wood forest? 0.0007642594
What is the total area of Cambridgeshire? 0.4027958125
What is the total area of the river Garw? 0.0040077796
How large is County Donegal? 0.6810451832000001
What is the size of Eretria? 0.0175444722
What is the area of West Sussex? 0.25898291500000004
What is the area of the Republic of Ireland? 9.458364665
What is the total area of Liverpool? 0.018048302000000002
What is the total area of Manchester? 0.0156526746
What is Londonderry's surface area? 945788.486
What is the total area of town Ballinasloe? 0.066345633
What is the total area of Greece? 13.7995223574
What area does Greenwich Park cover? 9.61285e-05
What is the total area of the United Kingdom? 33.7948826309
What is the total area of Arkansas? 13.5812443819
What is the size of Greece? 13.7995223574
How large is Newport? 0.028250078
How large is Trichonida? 0.0096175887
What is the total surface area o

In [118]:
places_to_countries = {
    "County Galway": "Ireland",
    "Glengarra Wood forest": "Ireland",
    "Cambridgeshire": "England",
    "river Garw": "Ireland",
    "County Donegal": "Ireland",
    "Eretria": "Greece",
    "West Sussex": "England",
    "Republic of Ireland": "Ireland",
    "Liverpool": "England",
    "Manchester": "England",
    "Londonderry": "Ireland",
    "Ballinasloe": "Ireland",
    "Greece": "Greece",
    "Greenwich Park": "England",
    "United Kingdom": "England",
    "Arkansas": "USA",
    "Newport": "England",
    "Trichonida": "Greece",
    "Andover Ponds": "USA",
    "Wells County, North Dakota": "USA",
    "Dundee": "England",
    "Oklahoma": "USA",
    "George Washington Park": "USA",
    "County Cavan": "Ireland",
    "Staffordshire": "England",
    "Cork City": "Ireland",
    "Texas": "USA",
    "England": "England",
    "Nevada": "USA",
    "Ireland": "Ireland"
}

In [119]:
mean_latitudes = {
    "USA": 39.0,
    "England": 52.5,
    "Ireland": 53.5,
    "Greece": 39.0
}

In [120]:
def change_degrees_in_km(row):
    if row['answer_type'][0] == 'sq degrees':
        values = row['cleaned_answer']
        q = row['question']
        country = None
        for k in places_to_countries:
            if k in q:
                country = places_to_countries[k]
                break

        if country is None:
            print("--------------------Problem---------------------")
        km2_par_deg2 = 111.32 * 111.32 * math.cos(math.radians(mean_latitudes[country]))
        n_values = []
        for v in values:
            try:
                fv = float(v)
                n_values.append(km2_par_deg2 * fv)
            except:
                continue
        row['cleaned_answer'] = n_values
        row['answer_type'] = ['sq km']*len(n_values)
        if n_values:
            print(row['question'], n_values[0], row['answer_type'])
        else:
            print('Error')
    return row

data = data.apply(change_degrees_in_km, axis=1)

What is the total area of County Galway? 50.30979304869696 ['sq km']
What is the total area of Glengarra Wood forest? 5.633454379385819 ['sq km']
What is the total area of Cambridgeshire? 3038.6345397392547 ['sq km', 'sq km']
What is the total area of the river Garw? 29.541859137399086 ['sq km']
How large is County Donegal? 5020.071679664859 ['sq km']
What is the size of Eretria? 168.96209952348778 ['sq km', 'sq km']
What is the area of West Sussex? 1953.7304169996942 ['sq km', 'sq km']
What is the area of the Republic of Ireland? 69718.82301201965 ['sq km']
What is the total area of Liverpool? 136.1538331306388 ['sq km']
What is the total area of Manchester? 118.08155944734237 ['sq km', 'sq km']
What is Londonderry's surface area? 6971528630.762518 ['sq km']
What is the total area of town Ballinasloe? 489.04219794610873 ['sq km']
What is the total area of Greece? 132896.34725675097 ['sq km']
What area does Greenwich Park cover? 0.725179784120335 ['sq km']
What is the total area of the

In [121]:
print(data[data['question'] == "What is the total area of County Galway?"]['cleaned_answer'])

5    [50.30979304869696]
Name: cleaned_answer, dtype: object


## Translating place names from Greek to English

In [122]:
data.loc[data['original_answer'].str.contains('gagentity', case=False, na=False), 'trad'] = 'True'
print(len(data[data['trad'] == 'True']))


7


In [123]:
to_trad = data[data['trad'] == 'True']
def print_row(row):
    print(row['question'])
    print(row['cleaned_answer'])
    print("\n")

to_trad.apply(print_row, axis=1)

Which regional units border with Thessaly?
['ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ ΠΙΕΡΙΑΣ', 'ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ ΓΡΕΒΕΝΩΝ', 'ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ ΚΟΖΑΝΗΣ', 'ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ ΑΙΤΩΛΟΑΚΑΡΝΑΝΙΑΣ', 'ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ ΦΘΙΩΤΙΔΑΣ', 'ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ ΕΥΡΥΤΑΝΙΑΣ']


Which regional units border the regional unit of Thessaloniki?
['ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ ΣΕΡΡΩΝ', 'ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ ΠΕΛΛΑΣ', 'ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ ΚΙΛΚΙΣ', 'ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ ΗΜΑΘΙΑΣ', 'ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ ΧΑΛΚΙΔΙΚΗΣ']


Which are Thessaly's neighboring Regional Units?
['ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ ΠΙΕΡΙΑΣ', 'ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ ΓΡΕΒΕΝΩΝ', 'ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ ΚΟΖΑΝΗΣ', 'ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ ΑΙΤΩΛΟΑΚΑΡΝΑΝΙΑΣ', 'ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ ΦΘΙΩΤΙΔΑΣ', 'ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ ΕΥΡΥΤΑΝΙΑΣ']


Which regional unit in the region of Epirus has a lake?
['ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ ΙΩΑΝΝΙΝΩΝ', 'ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ ΠΡΕΒΕΖΑΣ']


Which regional unit in the region of Thessaly has a beach?
['ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ ΣΠΟΡΑΔΩΝ', 'ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ ΜΑΓΝΗΣΙΑΣ']


Which r

286    None
326    None
391    None
475    None
547    None
576    None
604    None
dtype: object

In [124]:
a = "['REGIONAL UNIT OF PIERIA', 'REGIONAL UNIT OF GREVENA', 'REGIONAL UNIT OF KOZANI', 'REGIONAL UNIT OF ETOLOAKARNANIA', 'REGIONAL UNIT OF FTHIOTIDA', 'REGIONAL UNIT OF EVRYTANIA']"
a = ast.literal_eval(a.replace("REGIONAL UNIT OF ", ''))
data.at[data.index[data['question'] == 'Which regional units border with Thessaly?'][0], 'cleaned_answer'] = a

In [125]:
a = "['REGIONAL UNIT OF SERRA', 'REGIONAL UNIT OF PELLA', 'REGIONAL UNIT OF KILKIS', 'REGIONAL UNIT OF IMATHIA', 'REGIONAL UNIT OF HALKIDIKI']"
a = ast.literal_eval(a.replace("REGIONAL UNIT OF ", ''))
data.at[data.index[data['question'] == 'Which regional units border the regional unit of Thessaloniki?'][0], 'cleaned_answer'] = a

In [126]:
a = "['REGIONAL UNIT OF PIERIA', 'REGIONAL UNIT OF GREVENA', 'REGIONAL UNIT OF KOZANI', 'REGIONAL UNIT OF ETOLOAKARNANIA', 'REGIONAL UNIT OF FTHIOTIDA', 'REGIONAL UNIT OF EVRYTANIA']"
a = ast.literal_eval(a.replace("REGIONAL UNIT OF ", ''))
data.at[data.index[data['question'] == "Which are Thessaly's neighboring Regional Units?"][0], 'cleaned_answer'] = a

In [127]:
a = "['REGIONAL UNIT OF IOANNINA', 'REGIONAL UNIT OF PREVEZA']"
a = ast.literal_eval(a.replace("REGIONAL UNIT OF ", ''))
data.at[data.index[data['question'] == "Which regional unit in the region of Epirus has a lake?"][0], 'cleaned_answer'] = a

In [128]:
a = "['SPORADES REGIONAL UNIT', 'MAGNESIA REGIONAL UNIT']"
a = ast.literal_eval(a.replace("REGIONAL UNIT OF ", ''))
data.at[data.index[data['question'] == "Which regional unit in the region of Thessaly has a beach?"][0], 'cleaned_answer'] = a

In [129]:
a = "['REGIONAL UNIT OF SERRA', 'REGIONAL UNIT OF PIERIA', 'REGIONAL UNIT OF KILKIS', 'REGIONAL UNIT OF KASTORIA', 'REGIONAL UNIT OF FLORINA', 'REGIONAL UNIT OF PREVEZA', 'REGIONAL UNIT OF IOANNINA', 'REGIONAL UNIT OF TRIKALA', 'REGIONAL UNIT OF ETOLOAKARNANIA', 'REGIONAL UNIT OF FOKIDA', 'REGIONAL UNIT OF VIOTIA']"
a = ast.literal_eval(a.replace("REGIONAL UNIT OF ", ''))
data.at[data.index[data['question'] == "Which regional units of Greece share a border with three regional units?"][0], 'cleaned_answer'] = a

In [130]:
a = "['SPORADES REGIONAL UNIT', 'ITHAKI REGIONAL UNIT', 'IKARIAS REGIONAL UNIT', 'Perifereiakí Enótita Mykónou', 'Perifereiakí Enótita Tínou', 'Perifereiakí Enótita Karpáthou', 'Perifereiakí Enótita Kéas - Kýthnou', 'Perifereiakí Enótita Mílou']"
a = ast.literal_eval(a.replace("REGIONAL UNIT OF ", ''))
data.at[data.index[data['question'] == "Which Greek regional units have less than 10000 inhabitants?"][0], 'cleaned_answer'] = a

In [131]:
to_trad = data[data['trad'] == 'True']
def print_row(row):
    print(row['question'])
    print(row['cleaned_answer'])
    print("\n")

to_trad.apply(print_row, axis=1)

Which regional units border with Thessaly?
['PIERIA', 'GREVENA', 'KOZANI', 'ETOLOAKARNANIA', 'FTHIOTIDA', 'EVRYTANIA']


Which regional units border the regional unit of Thessaloniki?
['SERRA', 'PELLA', 'KILKIS', 'IMATHIA', 'HALKIDIKI']


Which are Thessaly's neighboring Regional Units?
['PIERIA', 'GREVENA', 'KOZANI', 'ETOLOAKARNANIA', 'FTHIOTIDA', 'EVRYTANIA']


Which regional unit in the region of Epirus has a lake?
['IOANNINA', 'PREVEZA']


Which regional unit in the region of Thessaly has a beach?
['SPORADES REGIONAL UNIT', 'MAGNESIA REGIONAL UNIT']


Which regional units of Greece share a border with three regional units?
['SERRA', 'PIERIA', 'KILKIS', 'KASTORIA', 'FLORINA', 'PREVEZA', 'IOANNINA', 'TRIKALA', 'ETOLOAKARNANIA', 'FOKIDA', 'VIOTIA']


Which Greek regional units have less than 10000 inhabitants?
['SPORADES REGIONAL UNIT', 'ITHAKI REGIONAL UNIT', 'IKARIAS REGIONAL UNIT', 'Perifereiakí Enótita Mykónou', 'Perifereiakí Enótita Tínou', 'Perifereiakí Enótita Karpáthou', 'Peri

286    None
326    None
391    None
475    None
547    None
576    None
604    None
dtype: object

In [132]:
data = data.rename(columns={'cleaned_answer':'answer'})

In [133]:
data.to_json(output_file, orient="records", force_ascii=False, indent=4)

## Converting to HF dataset

In [134]:
data.head()

,question,answer,category,original_answer,answer_type,question_id,trad
0,What is the population of Central Greece?,[570922.0],A,population\r\n570922\r\n,[numeric],1,NaN
1,Where is the Dorset county located?,"[[50.8054069086, -2.3225184771]]",A,None,[coord],2,NaN
2,Where is Lough Ramor located?,"[[53.8184256149, -7.0822994976]]",A,None,[coord],3,NaN
3,What is the population of the Municipality of ...,[38116.0],A,population\r\n38116\r\n,[numeric],4,NaN
4,Where is Oxfordshire located?,"[[51.7657655773, -1.3223245468]]",A,None,[coord],5,NaN


In [135]:
to_keep = ['question_id', 'question', 'answer', 'answer_type']
def get_cat(row):
    t = row['answer_type']
    if t[0] == 'sq km':
        t[0] = 'numeric'
    row['cat'] = t[0]
    return row

data = data.apply(get_cat, axis=1)

print(data['cat'].value_counts())

yesno = data[data["cat"] == "bool"][to_keep]

coord = data[data["cat"] == "coord"][to_keep]

place = data[data["cat"] == "str"][to_keep]

regression = data[data["cat"] == "numeric"][to_keep]

cat
str        455
numeric    231
bool       181
coord       87
Name: count, dtype: int64


In [136]:
def is_not_float(x):
    try:
        float(x)
        return False
    except (ValueError, TypeError):
        return True 

regression["answer"] = regression["answer"].apply(
    lambda lst: [float(x) for x in lst if not is_not_float(x)]
)

place["answer"] = place["answer"].apply(
    lambda lst: [str(x) for x in lst]
)

In [137]:
map_cat_to_data = {
    'YN': yesno,
    'coord': coord,
    'regression': regression,
    'place': place
}

d1 = {}

for name, data_csv in map_cat_to_data.items():
    test = datasets.Dataset.from_pandas(data_csv).remove_columns("__index_level_0__")
    d1[name] = datasets.DatasetDict({
            "test": test
        })
    print(d1[name]['test'].features)
    d1[name].save_to_disk(os.path.join(output_path, f"GeoQuestions1089_{name}"))

{'question_id': Value('int64'), 'question': Value('string'), 'answer': List(Value('bool')), 'answer_type': List(Value('string'))}


Saving the dataset (1/1 shards): 100%|██████████| 181/181 [00:00<00:00, 29051.32 examples/s]


{'question_id': Value('int64'), 'question': Value('string'), 'answer': List(List(Value('float64'))), 'answer_type': List(Value('string'))}


Saving the dataset (1/1 shards): 100%|██████████| 87/87 [00:00<00:00, 8259.68 examples/s]


{'question_id': Value('int64'), 'question': Value('string'), 'answer': List(Value('float64')), 'answer_type': List(Value('string'))}


Saving the dataset (1/1 shards): 100%|██████████| 231/231 [00:00<00:00, 40573.04 examples/s]


{'question_id': Value('int64'), 'question': Value('string'), 'answer': List(Value('string')), 'answer_type': List(Value('string'))}


Saving the dataset (1/1 shards): 100%|██████████| 455/455 [00:00<00:00, 19986.89 examples/s]


## Sanity Check

In [140]:
import hashlib
import json
# We check that the content of the dataset is the same
def content_hash(ds):
    h = hashlib.sha256()
    for ex in ds:
        h.update(json.dumps(ex, sort_keys=True).encode())
    return h.hexdigest()

for name in d1.keys():
    d2 = datasets.load_dataset("rfr2003/Geo_Benchmark", f"GeoQuestions1089_{name}")
    assert content_hash(d1[name]['test']) == content_hash(d2['test'])

Generating test split: 100%|██████████| 455/455 [00:00<00:00, 12476.85 examples/s]
